# Refresh: gold_source_coverage

Rebuilds `genealogy.gold_source_coverage` (citation grain: person, source_title, citation_date, ancestry_record_id) with `TRUNCATE` + `INSERT` using an explicit column list. Extracted from `notebook_01_document_matching` (Cell 8) so it can be re-run on its own: the rest of that notebook truncates and reloads `silver_document_person` and the document-match exceptions, which a coverage refresh must not do.

Contains **no view DDL**. Prerequisites: `silver_document_person` populated, `v_source_doctype_map` (a view over the hand-maintained `ref_source_doctype_map`) and `gold_person_life` exist, `gold_person_event_timeline` current.

Re-run after anything that changes inputs: `07_materialise_gold_tables` (birth/death year feeds the `citation_year` fallback, +/-2 year tolerance), document matching, new OCR transcripts.

Unmapped titles fall to `OTHER`; list them in `gold_housekeeping_unmapped_source_titles` and add mappings to `ref_source_doctype_map` (INSERT, not a notebook edit). The category-to-file join is case-sensitive on purpose: do not lower-case it, or every unmapped (`OTHER`) citation would match any `Other` file. Map a title to the file's exact `doc_type_detected` (e.g. `Other`) instead.

- 2026-10-16: lookup moved from a hardcoded VALUES list to `ref_source_doctype_map` (Asana 1218767063521380).


In [0]:
%sql
TRUNCATE TABLE genealogy.gold_source_coverage;

INSERT INTO genealogy.gold_source_coverage (
  person_gedcom_id, source_title, source_category,
  ancestry_record_id, citation_date, citation_text, citation_url,
  has_document, has_transcript, matched_file_ids, coverage_status, refreshed_at
)
WITH

event_person AS (
  SELECT DISTINCT ep.event_id, ep.person_id AS person_gedcom_id
  FROM genealogy.silver_event_participant ep
  WHERE ep.data_source = 'recorded' --exclude derived parentage events
),

citations AS (
  -- One row per (person, source_xref, citation_date, ancestry_record_id).
  -- ancestry_record_id = _APID from Ancestry, identifying a specific record
  -- within a collection. For family records (marriages), husband_id or wife_id
  -- is used instead of person_id. For non-Ancestry sources, ancestry_record_id
  -- is NULL and all citations for the same source_title should group by citation_date to allow for a person with multiple certificate citations (e.g. themselves and their child).
  --
  -- citation_year fallback chain:
  -- 1. 4-digit year extracted from citation_date (free text e.g. "21 Jan 1857", "Dec 1856")
  -- 2. 4-digit year extracted from citation_text (often contains event year)
  --    NOTE: citation_text may be truncated due to line breaks in source data —
  --    this is a known data pipeline issue to fix upstream in the GEDCOM ingestion.
  --    When truncated, this fallback returns NULL and the chain continues.
  --    Exclude Census as the text can contain estimate birth date of the person and not the census year
  -- 3. Title regex — Census only. For all other source types the title gives the
  --    collection date range (e.g. 1530 from "1530-1906") not the event year,
  --    so is deliberately excluded for non-Census categories.
  -- 4. birth_year / death_year from gold_person_life — last resort for parish
  --    registers where citation_date and citation_text contain no year.
  SELECT
    ep.person_gedcom_id,
    ss.source_title,
    COALESCE(dm.doc_type_category, 'OTHER')                              AS source_category,
    COALESCE(
      TRY_CAST(REGEXP_EXTRACT(MAX(es.citation_date), '(\\d{4})') AS INT),
      -- Text regex for non-Census
      CASE WHEN COALESCE(dm.doc_type_category, 'OTHER') <> 'Census'
        THEN TRY_CAST(REGEXP_EXTRACT(MAX(es.citation_text), '(\\d{4})') AS INT)
      END,
      -- Title regex only for Census — gives collection range for all other types
      CASE WHEN COALESCE(dm.doc_type_category, 'OTHER') = 'Census'
        THEN TRY_CAST(REGEXP_EXTRACT(ss.source_title, '(\\d{4})') AS INT)
      END,
      CASE
        WHEN COALESCE(dm.doc_type_category, 'OTHER')
             IN ('BirthCertificate', 'BaptismRegister', 'BirthIndex')
        THEN pl.birth_year
        WHEN COALESCE(dm.doc_type_category, 'OTHER')
             IN ('DeathCertificate', 'BurialRegister', 'DeathIndex')
        THEN pl.death_year
      END
    )                                                                    AS citation_year,
    COALESCE(es.citation_ancestry_person_id,
             es.citation_ancestry_husband_id,
             es.citation_ancestry_wife_id)                               AS ancestry_record_id,
    MAX(es.citation_date)                                                AS citation_date,
    MAX(es.citation_text)                                                AS citation_text,
    MAX(es.citation_url)                                                 AS citation_url
  FROM genealogy.silver_event_source es
  JOIN event_person ep ON es.event_id = ep.event_id
  LEFT JOIN genealogy.silver_source ss ON es.source_xref = ss.record_xref
  LEFT JOIN genealogy.v_source_doctype_map dm ON ss.source_title = dm.source_title
  LEFT JOIN genealogy.gold_person_life pl ON ep.person_gedcom_id = pl.person_gedcom_id
  GROUP BY
    ep.person_gedcom_id,
    ss.source_title,
    COALESCE(dm.doc_type_category, 'OTHER'),
    TRY_CAST(REGEXP_EXTRACT(ss.source_title, '(\\d{4})') AS INT),
    es.citation_date,
    COALESCE(es.citation_ancestry_person_id,
             es.citation_ancestry_husband_id,
             es.citation_ancestry_wife_id),
    pl.birth_year,
    pl.death_year
),

doc_matches AS (
  SELECT
    sdp.person_gedcom_id,
    ot.doc_type_detected,
    YEAR(TRY_CAST(ot.year AS DATE))    AS doc_year,
    sdp.file_id,
    ot.transcribed_text IS NOT NULL    AS has_transcript
  FROM genealogy.silver_document_person sdp
  JOIN genealogy.ocr_transcriptions ot ON sdp.file_id = ot.file_id
),

marriage_years AS (
  SELECT person_gedcom_id, event_year_parsed AS marriage_year
  FROM genealogy.gold_person_event_timeline
  WHERE event_type = 'MARR'
    AND event_year_parsed IS NOT NULL
),

doc_coverage AS (
  -- Match citations to held documents on person + doc type + year proximity.
  -- citation_year is derived from the fallback chain above so is the actual
  -- event year for most source types.
  -- Census uses the title-regex fallback year and requires exact match.
  -- Military, probate, electoral etc have no year guard.
  SELECT
    c.person_gedcom_id,
    c.source_category,
    c.citation_year,
    COUNT(DISTINCT dm.file_id) > 0                                      AS has_document,
    COUNT(DISTINCT CASE WHEN dm.has_transcript THEN dm.file_id END) > 0 AS has_transcript,
    CONCAT_WS(',', COLLECT_LIST(DISTINCT dm.file_id))                   AS matched_file_ids
  FROM citations c
  JOIN doc_matches dm
    ON  c.person_gedcom_id = dm.person_gedcom_id
    AND c.source_category  = dm.doc_type_detected
    AND (
      (c.source_category = 'Census'
          AND dm.doc_year = c.citation_year)
      OR (c.source_category IN ('BirthCertificate', 'BaptismRegister', 'BirthIndex')
          AND ABS(dm.doc_year - c.citation_year) <= 2)
      OR (c.source_category IN ('DeathCertificate', 'BurialRegister', 'DeathIndex')
          AND ABS(dm.doc_year - c.citation_year) <= 2)
      OR c.source_category NOT IN (
          'Census', 'BirthCertificate', 'BaptismRegister', 'BirthIndex',
          'MarriageCertificate', 'MarriageIndex',
          'DeathCertificate', 'BurialRegister', 'DeathIndex'
         )
    )
    GROUP BY c.person_gedcom_id, c.source_category, c.citation_year
  -- Marriage: join to all marriage years and match if doc within ±2 of any
  -- Having added extra filter to match doc year to citation year (to avoid false matches like father cited on child's marriage certificate), this union may be redundant. Unless it was added due to missing citation dates on some marriage sources - in which case these dates need to be added
  UNION ALL
  SELECT
    c.person_gedcom_id,
    c.source_category,
    c.citation_year,
    COUNT(DISTINCT dm.file_id) > 0                                      AS has_document,
    COUNT(DISTINCT CASE WHEN dm.has_transcript THEN dm.file_id END) > 0 AS has_transcript,
    CONCAT_WS(',', COLLECT_LIST(DISTINCT dm.file_id))                   AS matched_file_ids
  FROM citations c
  JOIN doc_matches dm
    ON  c.person_gedcom_id = dm.person_gedcom_id
    AND c.source_category  = dm.doc_type_detected
  JOIN marriage_years my
    ON  my.person_gedcom_id = c.person_gedcom_id
    AND ABS(dm.doc_year - my.marriage_year) <= 2 -- doc matches event
    AND dm.doc_year = c.citation_year -- doc matches citation
  WHERE c.source_category IN ('MarriageCertificate', 'MarriageIndex')
  GROUP BY c.person_gedcom_id, c.source_category, c.citation_year
)

SELECT
  c.person_gedcom_id, c.source_title, c.source_category,
  c.ancestry_record_id, c.citation_date, c.citation_text, c.citation_url,
  COALESCE(dc.has_document, FALSE)    AS has_document,
  COALESCE(dc.has_transcript, FALSE)  AS has_transcript,
  dc.matched_file_ids,
  CASE
    -- Index-only sources: no physical document to download
    WHEN c.source_category IN ('BirthIndex', 'MarriageIndex', 'DeathIndex') THEN 'INDEX_ONLY'
    WHEN c.source_category = 'OTHER_NO_FILE'                                THEN 'NOT_APPLICABLE'
    -- Non-Ancestry citations (no _APID) imply document held outside system
    WHEN c.ancestry_record_id IS NULL
         AND c.source_category = 'OTHER'                                    THEN 'NOT_APPLICABLE'
    WHEN dc.has_transcript = TRUE                                           THEN 'COVERED'
    WHEN dc.has_document   = TRUE                                           THEN 'DOCUMENT_NO_TRANSCRIPT'
    ELSE                                                                         'UNCOVERED'
  END AS coverage_status,
  CURRENT_TIMESTAMP()
FROM citations c
LEFT JOIN doc_coverage dc
  ON  c.person_gedcom_id = dc.person_gedcom_id
  AND c.source_category  = dc.source_category
  AND (
    -- Mirror the same year guard logic used in doc_coverage above
    (c.source_category = 'Census'
        AND c.citation_year = dc.citation_year)
    OR (c.source_category IN ('BirthCertificate', 'BaptismRegister', 'BirthIndex')
        AND ABS(dc.citation_year - c.citation_year) <= 2)
    OR c.source_category IN ('MarriageCertificate', 'MarriageIndex')
    OR (c.source_category IN ('DeathCertificate', 'BurialRegister', 'DeathIndex')
        AND ABS(dc.citation_year - c.citation_year) <= 2)
    OR c.source_category NOT IN (
        'Census', 'BirthCertificate', 'BaptismRegister', 'BirthIndex',
        'MarriageCertificate', 'MarriageIndex',
        'DeathCertificate', 'BurialRegister', 'DeathIndex'
       )
  );

## Summary
Coverage status counts after the refresh.

In [0]:
%sql
SELECT coverage_status, COUNT(*) AS n
FROM genealogy.gold_source_coverage sc
--LEFT JOIN genealogy.gold_ancestral_proximity ap ON sc.person_gedcom_id = ap.person_id
--WHERE ap.ancestral_proximity = 0
GROUP BY coverage_status
ORDER BY n DESC;